In [5]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
 
TRAIN_END = 2011
df = pd.read_csv("data/processed/targets.csv")
assert not df.duplicated(["company_name", "year"]).any()
 
df = df.drop(columns=["total_revenue"])          # identical to net_sales
df = df.drop(columns=[c for c in ["re_ta", "ebit_ta", "wc_ta", "mv_tl"]
                      if c in df.columns])       # helper cols from step 2
 
print("Spearman(net_income, ebit) =",
      round(df[["net_income", "ebit"]].corr(method="spearman").iloc[0, 1], 3),
      "(should be clearly positive)")
 
feat = pd.DataFrame(index=df.index)
GROUP = {}
 
 
def div(a, b):
    return (a / b.where(b > 0)).replace([np.inf, -np.inf], np.nan)
 
 
def add(name, s, group):
    feat[name] = s.replace([np.inf, -np.inf], np.nan)
    GROUP[name] = group
 
 
ca, cl, ta, tl = df.current_assets, df.current_liabilities, \
    df.total_assets, df.total_liabilities
sales = df.net_sales
 
# ---- liquidity
add("current_ratio", div(ca, cl), "liquidity")
add("quick_ratio", div(ca - df.inventory, cl), "liquidity")
add("wc_ta", div(ca - cl, ta), "liquidity")
# ---- leverage
add("tl_ta", div(tl, ta), "leverage")
add("ltd_ta", div(df.long_term_debt, ta), "leverage")
add("cl_ta", div(cl, ta), "leverage")
# ---- profitability
add("roa", div(df.net_income, ta), "profitability")
add("ebit_ta", div(df.ebit, ta), "profitability")
add("ebitda_ta", div(df.ebitda, ta), "profitability")
add("re_ta", div(df.retained_earnings, ta), "profitability")
add("ebit_margin", div(df.ebit, sales), "profitability")
add("ebitda_margin", div(df.ebitda, sales), "profitability")
add("gross_margin", div(df.gross_profit, sales), "profitability")
add("net_margin", div(df.net_income, sales), "profitability")
# ---- activity
add("asset_turnover", div(sales, ta), "activity")
add("inventory_ta", div(df.inventory, ta), "activity")
add("receivables_ta", div(df.total_receivables, ta), "activity")
add("receivables_sales", div(df.total_receivables, sales), "activity")
add("cogs_sales", div(df.cogs, sales), "activity")
add("opex_sales", div(df.operating_expenses, sales), "activity")
# ---- market / size
add("mv_tl", div(df.market_value, tl), "market_size")
add("mv_ta", div(df.market_value, ta), "market_size")
add("log_assets", np.log(ta.where(ta > 0)), "market_size")
add("log_sales", np.log(sales.where(sales > 0)), "market_size")
add("log_mv", np.log(df.market_value.where(df.market_value > 0)),
    "market_size")
 
train_mask = df.year <= TRAIN_END
bounds = {}
 
 
def winsorize(cols):
    for c in cols:
        lo, hi = feat.loc[train_mask, c].quantile([0.01, 0.99])
        bounds[c] = [float(lo), float(hi)]
        feat[c] = feat[c].clip(lo, hi)
 
 
winsorize(list(feat.columns))
 
# ---- calendar-aware changes (computed from winsorized levels)
D_BASE = ["current_ratio", "wc_ta", "tl_ta", "ltd_ta", "roa", "ebit_ta",
          "re_ta", "ebit_margin", "asset_turnover", "mv_tl"]
GROWTH = {"log_sales": "g_sales", "log_assets": "g_assets"}
 
 
def lag_values(cols, k):
    lag = df[["company_name", "year"]].copy()
    for c in cols:
        lag[c] = feat[c].values
    lag["year"] = lag["year"] + k
    lag = lag.rename(columns={c: f"{c}__lag" for c in cols})
    m = df[["company_name", "year"]].merge(
        lag, on=["company_name", "year"], how="left")
    m.index = df.index
    return m
 
 
new_cols = []
for k in (1, 2):
    cols = D_BASE + (list(GROWTH) if k == 1 else [])
    lag = lag_values(cols, k)
    for c in cols:
        name = GROWTH[c] if c in GROWTH else f"d{k}_{c}"
        feat[name] = feat[c] - lag[f"{c}__lag"]
        GROUP[name] = "dynamics"
        new_cols.append(name)
winsorize(new_cols)
 
# ---- Altman (1968) Z from winsorized ratios (baseline only, not a feature)
z = (1.2 * feat.wc_ta + 1.4 * feat.re_ta + 3.3 * feat.ebit_ta
     + 0.6 * feat.mv_tl + 1.0 * feat.asset_turnover)
 
meta = ["company_name", "year", "split", "failed_co", "years_to_end",
        "yrs_before_fail", "y1", "y2", "y3"]
out = pd.concat([df[meta], feat], axis=1)
out["altman_z"] = z
out.to_csv("data/processed/features.csv", index=False)
 
pd.Series(GROUP, name="group").rename_axis("feature").to_csv(
    "results/feature_groups.csv")
with open("results/winsor_bounds.json", "w") as f:
    json.dump(bounds, f, indent=1)
 
summ = pd.DataFrame({
    "group": pd.Series(GROUP),
    "train_missing_%": (feat[train_mask].isna().mean() * 100).round(2),
    "train_median": feat[train_mask].median().round(3)})
summ.to_csv("results/step3_feature_summary.csv")
print("\nFeatures:", feat.shape[1])
print(summ.groupby("group").size().to_string())
print("\nHighest train missingness:")
print(summ.sort_values("train_missing_%", ascending=False).head(8)
      .to_string())
 
# ---- Altman baseline sanity check (score = -Z, no fitting)
print("\n=== Altman Z baseline (risk score = -Z) ===")
rows = []
for s in ("train", "val", "test"):
    for h in (1, 2, 3):
        d = out[(out.split == s) & out.altman_z.notna()]
        y = d[f"y{h}"]
        rows.append({"split": s, "horizon": h, "n": len(d),
                     "pos": int(y.sum()),
                     "prevalence": round(float(y.mean()), 4),
                     "ROC_AUC": round(roc_auc_score(y, -d.altman_z), 3),
                     "PR_AUC": round(average_precision_score(
                         y, -d.altman_z), 3)})
base = pd.DataFrame(rows)
print(base.to_string(index=False))
base.to_csv("results/step3_altman_baseline.csv", index=False)
print("\nShare of rows with Z < 1.81 (classic distress zone):")
print("  y1 = 1 rows:", round(float((out[out.y1 == 1].altman_z < 1.81)
                                    .mean()), 3))
print("  never-failed rows:", round(float((out[out.failed_co == 0]
                                           .altman_z < 1.81).mean()), 3))
 

Spearman(net_income, ebit) = 0.822 (should be clearly positive)

Features: 47
group
activity          6
dynamics         22
leverage          3
liquidity         3
market_size       5
profitability     8

Highest train missingness:
                      group  train_missing_%  train_median
d2_ebit_margin     dynamics            27.15         0.002
d2_re_ta           dynamics            27.13         0.001
d2_mv_tl           dynamics            27.13        -0.083
d2_tl_ta           dynamics            27.13         0.004
d2_roa             dynamics            27.13        -0.001
d2_asset_turnover  dynamics            27.13         0.013
d2_wc_ta           dynamics            27.13        -0.000
d2_ltd_ta          dynamics            27.13         0.000

=== Altman Z baseline (risk score = -Z) ===
split  horizon     n  pos  prevalence  ROC_AUC  PR_AUC
train        1 55927  403      0.0072    0.761   0.019
train        2 55927  820      0.0147    0.732   0.032
train        3 55927 1243  

In [ ]:
pip install xgboost interpret shap